<a href="https://colab.research.google.com/github/nikamsudarshan/quality_control_vision_system/blob/main/Model_Training_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

# 1. Setup Kaggle credentials (prompts for upload only if missing)
if not os.path.exists('/root/.kaggle/kaggle.json'):
    from google.colab import files
    print("Please upload your kaggle.json file:")
    files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# 2. Download and extract the dataset
print("Downloading NEU Surface Defect Database...")
!kaggle datasets download -d kaustubhdikshit/neu-surface-defect-database
!unzip -q -o neu-surface-defect-database.zip -d neu_data
print("Data extracted successfully!")

Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/kaustubhdikshit/neu-surface-defect-database
License(s): unknown
100% 26.4M/26.4M [00:00<00:00, 134MB/s]

Data extracted successfully!


In [2]:
import glob
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 32

# 1. Auto-discover the image directory
print("Scanning extracted files to locate the dataset...")
images = glob.glob('neu_data/**/*.bmp', recursive=True) + glob.glob('neu_data/**/*.jpg', recursive=True)
DATA_DIR = os.path.dirname(os.path.dirname(images[0]))
print(f"Dataset root set to: {DATA_DIR}")

# 2. Load Datasets
raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training",
    seed=42, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE
)
raw_val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation",
    seed=42, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE
)
class_names = raw_train_ds.class_names
print(f"Classes found: {class_names}")

# 3. Feature Engineering: The Severity Proxy
def prepare_multi_output(images, labels):
    """Calculates the standard deviation of pixels to act as a severity proxy."""
    std_dev = tf.math.reduce_std(images, axis=[1, 2, 3])
    severity_score = tf.clip_by_value(std_dev / 80.0, 0.0, 1.0)
    severity_score = tf.expand_dims(severity_score, axis=-1)
    return images, (labels, severity_score)

# 4. Optimize the data loading pipeline
train_ds = raw_train_ds.map(prepare_multi_output).prefetch(tf.data.AUTOTUNE)
val_ds = raw_val_ds.map(prepare_multi_output).prefetch(tf.data.AUTOTUNE)
print("Data pipeline ready for training.")

Scanning extracted files to locate the dataset...
Dataset root set to: neu_data/NEU-DET/train/images
Found 1440 files belonging to 6 classes.
Using 1152 files for training.
Found 1440 files belonging to 6 classes.
Using 288 files for validation.
Classes found: ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
Data pipeline ready for training.


In [3]:
from tensorflow.keras import layers, Model

# 1. The Shared Feature Extractor (Frozen)
base_model = tf.keras.applications.EfficientNetB0(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)

# 2. Parallel Output Heads
class_output = layers.Dense(len(class_names), activation='softmax', name='class_output')(x)
severity_output = layers.Dense(1, activation='sigmoid', name='severity_output')(x)

# 3. Compile Model with Dual Loss Functions
model = Model(inputs=base_model.input, outputs=[class_output, severity_output], name="Vision_QC_System")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss={'class_output': 'sparse_categorical_crossentropy', 'severity_output': 'mse'},
    metrics={'class_output': 'accuracy', 'severity_output': 'mae'}
)

model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "Vision_QC_System"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,214,442 (16.08 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [4]:
print("Initiating Multi-Task Training Sequence...")
# Train the network for 10 epochs
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)
print("Training Complete!")

Initiating Multi-Task Training Sequence...
Epoch 1/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 119s 3s/step - class_output_accuracy: 0.7882 - class_output_loss: 0.6788 - loss: 0.7151 - severity_output_loss: 0.0363 - severity_output_mae: 0.1499 - val_class_output_accuracy: 0.9757 - val_class_output_loss: 0.1559 - val_loss: 0.1697 - val_severity_output_loss: 0.0137 - val_severity_output_mae: 0.0903
Epoch 2/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 123s 3s/step - class_output_accuracy: 0.9757 - class_output_loss: 0.1199 - loss: 0.1451 - severity_output_loss: 0.0252 - severity_output_mae: 0.1251 - val_class_output_accuracy: 0.9757 - val_class_output_loss: 0.0848 - val_loss: 0.0947 - val_severity_output_loss: 0.0099 - val_severity_output_mae: 0.0746
Epoch 3/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 104s 3s/step - class_output_accuracy: 0.9835 - class_output_loss: 0.0715 - loss: 0.0911 - severity_output_loss: 0.0196 - severity_output_mae: 0.1086 - val_class_output_accuracy: 0.9896 - val_class_output_loss: 0.0541 - val_loss: 0.06

In [5]:
# Export the trained AI brain
model_filename = 'vision_qc_model.keras'
model.save(model_filename)
print(f"Model successfully saved to {model_filename}. Ready for download!")

Model successfully saved to vision_qc_model.keras. Ready for download!
